<table>
    <tr>
        <td>
            <h1>Modèle de langage: Auto-complétion
        </td>
    </tr>
</table>


<center><i>Réalisé par : </i>Douba JAFUNO </center>

<table style="width: 100%">
<tr>
    <td style="width: 15%">
    </td>
    <td style="width: 70%; text-align:left">
        <a href="#1"><h1>I.Introduction</h1></a><br>
           &nbsp; <a href="#presentation">I.1 Présentation du problème</a><br><br>
           


<a href="#2"><h1>II. Chargement et pré-traitement des données</h1></a><br><br>
    &nbsp; <a href="#pd">II.1 Prétraitement des données</a><br>
    &nbsp; <a href="#dett">II.2 Diviser en ensembles de train et de test</a><br>
    &nbsp; <a href="#mmhv">II.3 Manipulation des mots «hors vocabulaire</a><br>
    &nbsp; <a href="#cv">II.4 Construction du vocabulaire</a><br>
    &nbsp; <a href="#pdtt">II.5 Prétraiter les données de train et test</a><br><br>

  

<a href="#3"><h1>III.Développer des modèles de langage basés sur le n-gramme</h1></a><br><br>
    &nbsp; <a href="#vd">III.1 Probabilité de séquence avec approximation</a><br>
    &nbsp; <a href="#idt">III.2 Modèle de langage N-gramme</a><br>
    &nbsp; <a href="#eptm">III.3 Estimer les probabilités pour tous les mots</a><br>
    &nbsp; <a href="#mcp">III.4 Matrices de comptage et de probabilité</a><br>
    &nbsp; <a href="#mlg">III.5 Modèle de langage Générative</a><br>
    &nbsp; <a href="#per">III.6 Perplexité</a><br>



<a href="#4"><h1>IV. Construire un système d'Auto-Complétion</h1></a><br><br>
    &nbsp; <a href="#rms">IV.1 Recevoir de multiples suggestions</a><br>
    &nbsp; <a href="#spmunlv">IV.2 Suggérer plusieurs mots en utilisant des n-grammes de longueur variable</a><br><br>

<a href="#5"><h1>V. Référence </h1></a><br><br>
   </td>
    <td style="width: 0%">
    </td>
</tr>
</table>


# <a name="1">I. Introduction</a>

## <a name="presentation"> Présentation du problème </a>

Un modèle de langue est un élément clé d'un système d'auto-complétion.
Un modèle de langage attribue la probabilité à une séquence de mots, de telle sorte que les séquences plus "probables" reçoivent des scores plus élevés.  Par exemple,
>"I am a pen"
devrait avoir une probabilité plus élevée que
>"I eat scrambled"
puisque la première semble être une phrase plus naturelle dans le monde réel.

Vous pouvez profiter de ce calcul de probabilité pour développer un système d'auto-complétion.  
Supposons que l'utilisateur ait tapé
>"I eat scrambled"
Ensuite, vous pouvez trouver un mot "x" tel que "je mange du x brouillé" reçoit la plus grande probabilité.  Si x = "oeufs", la phrase serait
>"I eat scrambled eggs"

Bien qu'une variété de modèles linguistiques ait été développée, cette mission utilise les **N-grammes**, une méthode simple mais puissante de modélisation linguistique.
- Les N-grammes sont également utilisés dans la traduction automatique et la reconnaissance vocale.


Voici les étapes de ce travail :

1. Charger et prétraiter les données
    - Charger et tokeniser les données.
    - Séparez les phrases en ensembles de train et de test.
    - Remplacez les mots à basse fréquence par un marqueur inconnu `<unk>`.
1. Développer des modèles de langage basés sur N-gram
    - Calculer le nombre de n-grammes à partir d'un ensemble de données donné.
    - Estimer la probabilité conditionnelle d'un mot suivant avec un lissage k.
1. Évaluer les modèles à n-grammes en calculant le score de perplexité.
1. Utilisez votre propre modèle pour suggérer un mot à venir compte tenu de votre phrase.

In [ ]:
import math
import random
import numpy as np
import pandas as pd
import nltk
nltk.data.path.append('.')

# <a name="2">II. Chargement et pré-traitement des données</a>

Nous utiliserons les données Twitter.
Chargez les données et affichez les premières phrases en exécutant la cellule suivante.

Notez que les données sont une longue chaîne contenant de nombreux tweets.
Observez qu'il y a un saut de ligne "\n" entre les tweets.

In [ ]:
with open("en_US.twitter.txt", "r", encoding="utf8") as f:
    data = f.read()
print("Type de données : ", type(data))
print("Nombre de lettres :", len(data))
print("Les 300 premières lettres des données")
print("-------")
print(data[0:300])
print("-------")

print("Les 300 dernières lettres des données")
print("-------")
print(data[-300 :])
print("-------")

Type de données :  <class 'str'>
Nombre de lettres : 3335477
Les 300 premières lettres des données
-------
How are you? Btw thanks for the RT. You gonna be in DC anytime soon? Love to see you. Been way, way too long.
When you meet someone special... you'll know. Your heart will beat more rapidly and you'll smile for no reason.
they've decided its more fun if I don't.
So Tired D; Played Lazer Tag & Ran A 
-------
Les 300 dernières lettres des données
-------
ust had one a few weeks back....hopefully we will be back soon! wish you the best yo
Colombia is with an 'o'...“: We now ship to 4 countries in South America (fist pump). Please welcome Columbia to the Stunner Family”
#GutsiestMovesYouCanMake Giving a cat a bath.
Coffee after 5 was a TERRIBLE idea.

-------


## <a name="pd"> Prétraitement des données</a>

Prétraitons ces données en suivant les étapes suivantes :

1. Divisons les données en phrases en utilisant "\n" comme délimiteur.
1. Divisons chaque phrase en token. Notez que dans cette tâche, nous utilisons indifféremment les termes "token" et "words".
1. Attribuons des phrases aux ensembles de train et de test.
1. Trouvons les tokens qui apparaissent au moins N fois dans les données de formation.
1. Remplaçons les tokens qui apparaissent moins de N fois par `<unk>` (mot peu fréquent).


Note : nous omettons les données de validation dans cet exercice.
- Dans les applications réelles, nous devrions tenir une partie des données comme un ensemble de validation et l'utiliser pour régler notre formation.
- Nous sautons ce processus pour des raisons de simplicité.

Divissons les données en phrases.

In [ ]:
def split_to_sentences(data):

    """
     Diviser les données par saut de ligne "\ n"

     Args:
         données: str

     Retour:
         Une liste de phrases
    """
    sentences = data.split('\n')



    # Effacement supplémentaire (Cette partie est déjà implémentée)
    # - Supprimer les espaces de début et de fin de chaque phrase
    # - Supprimez les phrases si ce sont des chaînes vides.
    sentences = [s.strip() for s in sentences]
    sentences = [s for s in sentences if len(s) > 0]

    return sentences

In [ ]:
# test
x = """
I have a pen.\nI have an apple. \nAh\nApple pen.\n
"""
print(x)

split_to_sentences(x)


I have a pen.
I have an apple. 
Ah
Apple pen.




['I have a pen.', 'I have an apple.', 'Ah', 'Apple pen.']


L'étape suivante consiste à tokeniser les phrases (diviser une phrase en une liste de mots).
- Convertissons tous les tokens en minuscules afin que les mots qui sont en majuscules (par exemple, au début d'une phrase) dans le texte d'origine soient traités de la même manière que les versions minuscules des mots.
- Ajoutons chaque liste de mots tokenisés dans une liste de phrases tokenisées.

In [ ]:

def tokenize_sentences(sentences):
    """
    Tokenize des phrases en token (mots)

     Args:
         sentences: liste de chaînes

     Retour:
         Liste des listes de jetons
    """

    # Initialiser la liste des listes de phrases tokenisées
    tokenized_sentences = []

    # Parcourez chaque phrase
    for sentence in sentences:

        # Convertir en lettres minuscules
        sentence = sentence.lower()

        # Convertir en une liste de mots avec word_tokenize
        tokenized = nltk.word_tokenize(sentence)

        # ajouter la liste de mots à la liste des listes
        tokenized_sentences.append(tokenized)


    return tokenized_sentences

In [ ]:
# test
sentences = ["Sky is blue.", "Leaves are green.", "Roses are red."]
tokenize_sentences(sentences)

[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green', '.'],
 ['roses', 'are', 'red', '.']]

Utilisons les deux fonctions que nous venons d'implémenter pour obtenir les données tokenisées.
- divisons les données en phrases
- tokenizons ces phrases

In [ ]:
def get_tokenized_data(data):
    """
    Faire une liste de phrases tokenisées

    Args:
        data: chaîne de caractère

    Retour:
        Liste des listes de tokens
    """
     # Obtenez les phrases en divissant les données
    sentences = split_to_sentences(data)

    # Obtenez la liste des listes de tokens en tokenisant les phrases
    tokenized_sentences = tokenize_sentences(sentences)


    return tokenized_sentences

In [ ]:
# test
x = "Sky is blue.\nLeaves are green\nRoses are red."
get_tokenized_data(x)

[['sky', 'is', 'blue', '.'],
 ['leaves', 'are', 'green'],
 ['roses', 'are', 'red', '.']]

## <a name="dett"> Diviser en ensembles de train et de test</a>

Exécutez maintenant la cellule ci-dessous pour diviser les données en ensembles d'entraînement et de test.

In [ ]:
tokenized_data = get_tokenized_data(data)
random.seed(87)
random.shuffle(tokenized_data)

train_size = int(len(tokenized_data) * 0.8)
train_data = tokenized_data[0:train_size]
test_data = tokenized_data[train_size:]

In [ ]:
print("{} les données sont divisées en ensemble de {} train et de {} de test ".format(
    len(tokenized_data), len(train_data), len(test_data)))

print("Premier échantillon d'entraînement:")
print(train_data[0])

print("Premier échantillon de test")
print(test_data[0])

47961 les données sont divisées en ensemble de 38368 train et de 9593 de test 
Premier échantillon d'entraînement:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']
Premier échantillon de test
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']


Nous n'utiliserons pas tous les tokens (mots) apparaissant dans les données pour l'entraînement. Au lieu de cela, nous utiliserons les mots les plus fréquemment utilisés.
- Nous nous concentrerons sur les mots qui apparaissent au moins N fois dans les données.
- Comptons d'abord le nombre de fois où chaque mot apparaît dans les données.

Nous aurons besoin d'une double boucle for, une pour les phrases et l'autre pour les tokens dans une phrase.

In [ ]:
def count_words(tokenized_sentences):
    """
     Compte le nombre d'apparitions de mots dans les phrases tokenisées

    Args:
        tokenized_sentences: Liste des listes de chaînes

    Retour:
        dict qui indique le mot (str) à la fréquence (int)
    """

    word_counts = {}


    # Boucle sur chaque phrase
    for sentence in tokenized_sentences:

        # Parcourer chaque token de la phrase
        for token in sentence: #

            # Si le token n'est pas encore dans le dictionnaire, définisser le nombre sur 1
            if token not in word_counts.keys():
                word_counts[token] = 1

            # Si le token est déjà dans le dictionnaire, incrémenter le nombre de 1
            else:
                word_counts[token] += 1


    return word_counts

In [ ]:
# test
tokenized_sentences = [['sky', 'is', 'blue', '.'],
                       ['leaves', 'are', 'green', '.'],
                       ['roses', 'are', 'red', '.']]
count_words(tokenized_sentences)

{'sky': 1,
 'is': 1,
 'blue': 1,
 '.': 3,
 'leaves': 1,
 'are': 2,
 'green': 1,
 'roses': 1,
 'red': 1}

## <a name="mmhv"> Manipulation des mots «hors vocabulaire</a>

Si notre modèle effectue une saisie semi-automatique, mais rencontre un mot qu'il n'a jamais vu pendant l'entraînement, il n'aura pas de mot d'entrée pour l'aider à déterminer le mot suivant à suggérer. Le modèle ne pourra pas prédire le mot suivant car il n'y a pas de décompte pour le mot actuel.
- Ce «nouveau» mot est appelé un «unknown word» (mot inconnu), ou des mots <b> hors vocabulaire (OOV) </b>.
- Le pourcentage de mots inconnus dans l'ensemble de test est appelé taux <b> OOV </b>.

Pour gérer les mots inconnus lors de la prédiction, utilisons un token spécial pour représenter tous les mots inconnus «unk».
- Modifions les données d'entraînement afin qu'elles contiennent des mots «unknown» sur lesquels s'entraîner.
- Les mots à convertir en mots "unknown" sont ceux qui n'apparaissent pas très fréquemment dans l'ensemble d'apprentissage.
- Créons une liste des mots les plus fréquents de l'ensemble d'apprentissage, appelée <b> vocabulaire fermé </b>.
- Convertissons tous les autres mots <b> (vocabulaire ouvert </b>) qui ne font pas partie du vocabulaire fermé en token «unk» celà nous aidera aussi pour nos calcul de probabilité.

## <a name="cv"> Construction du vocabulaire</a>


Nous allons créer une fonction qui prend en compte un document texte et un seuil "count_threshold".
- Tout mot dont le nombre est supérieur ou égal au seuil "count_threshold" est conservé dans le vocabulaire fermé.
- Si nous utilisons la fonction "count_threshold" pour un mot que nous souhaitons conserver, nous obtenons le document contenant uniquement le mot "vocabulaire fermé" et le mot "unk".


In [ ]:
### GRADED_FUNCTION: get_words_with_nplus_frequency ###
def get_words_with_nplus_frequency(tokenized_sentences, count_threshold):
    """
    Trouvez les mots qui apparaissent N fois ou plus

     Args:
         tokenized_sentences: Liste des listes de phrases
         count_threshold: nombre minimum d'occurrences pour qu'un mot soit dans le vocabulaire fermé.

     Retour:
         Liste des mots qui apparaissent N fois ou plus
    """
    # Initialise une liste vide pour contenir les mots qui
    # apparaît au moins fois 'minimum_freq'.
    closed_vocab = []

    # Obtener le nombre de mots des phrases tokenisées
    # Utiliser la fonction que nous avons définie précédemment pour compter les mots
    word_counts = count_words(tokenized_sentences)

    # pour chaque mot et son décompte
    for word, cnt in word_counts.items():

        # vérifier que le nombre de mots
        # est au moins aussi grand que le nombre minimum
        if cnt >= count_threshold:

            # ajouter le mot à la liste
            closed_vocab.append(word)

    return closed_vocab

In [ ]:
# test
tokenized_sentences = [['sky', 'is', 'blue', '.'],
                       ['leaves', 'are', 'green', '.'],
                       ['roses', 'are', 'red', '.']]
tmp_closed_vocab = get_words_with_nplus_frequency(tokenized_sentences, count_threshold=2)
print(f"Vocabulaire fermé:")
print(tmp_closed_vocab)

Vocabulaire fermé:
['.', 'are']


Les mots qui apparaissent plusieurs fois "count_threshold" ou plus sont dans le "vocabulaire fermé".
- Tous les autres mots sont considérés comme "inconnus".
- Remplaçons les mots qui ne font pas partie du vocabulaire fermé par le token"<unk\>".

In [ ]:
def replace_oov_words_by_unk(tokenized_sentences, vocabulary, unknown_token="<unk>"):
    """
    Remplacez les mots qui ne font pas partie du vocabulaire donné par le jeton «».

    Args:
        tokenized_sentences: Liste des listes de chaînes
        vocabulaire: Liste des chaînes que nous utiliserons
        unknown_token: Une chaîne représentant des mots inconnus (hors vocabulaire)

    Retour:
        Liste des listes de chaînes, avec les mots ne faisant pas partie du vocabulaire remplacés
    """

    # Placez le vocabulaire dans un ensemble pour une recherche plus rapide
    vocabulary = set(vocabulary)

    ## Initialise une liste qui contiendra les phrases
    # après que les mots moins fréquents sont remplacés par le token inconnu
    replaced_tokenized_sentences = []

    # Parcourir chaque phrase
    for sentence in tokenized_sentences:

        # Initialise la liste qui contiendra
        # une seule phrase avec des remplacements "unknown_token"
        replaced_sentence = []

       # pour chaque token de la phrase
        for token in sentence:

            # Vérifiez si le token est dans le vocabulaire fermé
            if token in vocabulary:
                # Si tel est le cas, ajouter le mot à la phrase remplacée
                replaced_sentence.append(token)
            else:
                # sinon, ajouter le token inconnu à la place
                replaced_sentence.append(unknown_token)


        # Ajouter la liste des tokens à la liste des listes
        replaced_tokenized_sentences.append(replaced_sentence)

    return replaced_tokenized_sentences

In [ ]:
tokenized_sentences = [["dogs", "run"], ["cats", "sleep"]]
vocabulary = ["dogs", "sleep"]
tmp_replaced_tokenized_sentences = replace_oov_words_by_unk(tokenized_sentences, vocabulary)
print(f"Phrase originale :")
print(tokenized_sentences)
print(f"Phrases tokénissées avec des mots moins fréquents convertis en '<unk>':")
print(tmp_replaced_tokenized_sentences)

Phrase originale :
[['dogs', 'run'], ['cats', 'sleep']]
tokenized_sentences avec des mots moins fréquents convertis en '<unk>':
[['dogs', '<unk>'], ['<unk>', 'sleep']]


Nous sommes maintenant prêts à traiter nos données en combinant les fonctions que nous venons d'implémenter.

1. Recherchons les tokens qui apparaissent au moins count_threshold fois dans les données d'entraînement.
1. Remplaçons les tokens qui apparaissent moins que count_threshold fois par "<unk \>" à la fois pour les données d'entraînement et de test.

In [ ]:
def preprocess_data(train_data, test_data, count_threshold):
    """
    Prétraiter les données, c'est-à-dire
        - Trouvez les tokens qui apparaissent au moins N fois dans les données d'entraînement.
        - Remplacez les tokens qui apparaissent moins de N fois par "" à la fois pour les données d'entraînement et de test.
    Args:
        train_data, test_data: Liste des listes de chaînes.
        count_threshold: les mots dont le nombre est inférieur à cela sont
                      traité comme inconnu.

    Retour:
        Tuple de
        - données d'entraînement avec des mots peu fréquents remplacés par ""
        - données de test avec des mots peu fréquents remplacés par ""
        - vocabulaire des mots qui apparaissent n fois ou plus dans les données d'apprentissage
    """
    ### START CODE HERE (Replace instances of 'None' with your code) ###

    # Obtenez le vocabulaire fermé en utilisant les données du train
    vocabulary = get_words_with_nplus_frequency(train_data,count_threshold)

    # Pour les données du train, remplacez les mots moins courants par "<unk>"
    train_data_replaced = replace_oov_words_by_unk(train_data,vocabulary)

    # Pour les données du test, remplacez les mots moins courants par "<unk>"
    test_data_replaced = replace_oov_words_by_unk(test_data,vocabulary)

    ### END CODE HERE ###
    return train_data_replaced, test_data_replaced, vocabulary

In [ ]:
# test
tmp_train = [['sky', 'is', 'blue', '.'],
     ['leaves', 'are', 'green']]
tmp_test = [['roses', 'are', 'red', '.']]

tmp_train_repl, tmp_test_repl, tmp_vocab = preprocess_data(tmp_train,
                                                           tmp_test,
                                                           count_threshold = 1)

print("tmp_train_repl")
print(tmp_train_repl)
print()
print("tmp_test_repl")
print(tmp_test_repl)
print()
print("tmp_vocab")
print(tmp_vocab)

tmp_train_repl
[['sky', 'is', 'blue', '.'], ['leaves', 'are', 'green']]

tmp_test_repl
[['<unk>', 'are', '<unk>', '.']]

tmp_vocab
['sky', 'is', 'blue', '.', 'leaves', 'are', 'green']


## <a name="pdtt">Prétraiter les données de train et test</a>

Exécutons la cellule ci-dessous pour terminer le prétraitement pour les ensembles de train et de test.

In [ ]:
minimum_freq = 2
train_data_processed, test_data_processed, vocabulary = preprocess_data(train_data,
                                                                        test_data,
                                                                        minimum_freq)

In [ ]:
print("Premier échantillon d'entraînement ou de formation prétraité:")
print(train_data_processed[0])
print()
print("Premier échantillon de test prétraité:")
print(test_data_processed[0])
print()
print("10 premiers vocabulaires:")
print(vocabulary[0:10])
print()
print("Size of vocabulary:", len(vocabulary))

Premier échantillon d'entraînement ou de formation prétraité:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the', 'team', 'local', 'company', 'and', 'quality', 'production']

Premier échantillon de test prétraité:
['that', 'picture', 'i', 'just', 'seen', 'whoa', 'dere', '!', '!', '>', '>', '>', '>', '>', '>', '>']

10 premiers vocabulaires:
['i', 'personally', 'would', 'like', 'as', 'our', 'official', 'glove', 'of', 'the']

Size of vocabulary: 14821


Nous avons terminé avec la section de prétraitement.
Les objets `train_data_processed`,` test_data_processed` et `vocabulary` seront utilisés dans la suite.

<a name='2'></a>
# <a name="3">III. Développer des modèles de langage basés sur le n-gram</a>



N-gramme: c'est une séquence de N-mots (notamment la ponctuation) qui se suivent.

Exemple: Soit un corpus: 'I am happy because I am learning'
- unigramme: {I, am, happy, because,learning}
- bigramme: {I am, am happy, happy because, because learning}  
- trigramme: {I am happy, am happy because,....}

avec un grand corpus $m =500$ on note $w_1^m=w_1..........w_m$  dans le corpus  tout les mot du premier au $m^{ème}$ mot et $w_t^m$ du $t^{ème}$  mot au $m^{ème}$ .

Proba unigram  $$P(w_N)=\frac{C(w_N)}{m}$$
- La fonction $C(w_N)$ indique le nombre d'occurrences de la séquence de mot $w_N$ dans la séquence donnée(du corpus).
- ici $m=7$ car 7 mots unique dans notre corpus et on a $P(happy)=\frac{1}{7}$  et $P(I)=\frac{2}{7}$

Proba bigram
- $P(am|I) = \frac{C(\text{I am})}{C(I)}=\frac{2}{2}=1$
- $P(happy|I)=0$

Proba trigram
- $P(happy|I am) = \frac{C(\text{I  am  happy})}{C(I am)}=\frac{1}{2}$


On a au final $$P(w_N|w_1^{N-1})=\frac{C(w_1^{N-1}w_N)}{C(w_1^{N-1})}=\frac{C(w_1^{N})}{C(w_1^{N-1})}$$




## <a name="psa">Probabilité de séquence avec approximation </a>

$$P(B|A)=\frac{P(A,B)}{P(A)} \Rightarrow  P(A,B)= P(A)P(B|A)$$

En généralisant on a $$P(A,B,C,D)= P(A)P(B|A)P(C|A,B)P(D|A,B,C)$$

Exemple the teacher drinks tea
   
   $$P(tea|\text{the teacher drinks})=\frac{C(\text{the teacher drinks tea})}{C(\text{the teacher drinks})}$$
 Pour calculer $C(\text{the teacher drinks tea})$ on voit que plus la phrase est longue moins il ya de chance qu'elle soit dans le corpus....la solution est de considérer juste l'avant dernier mot tel que $$P(tea|\text{the teacher drinks})\approx P(tea|\text{drinks})$$ il s'agit d'une approximation de la probabilité de la séquence
 d'ou
   $$P(A,B,C,D)= P(A)P(B|A)P(C|B)P(D|C)$$


Voici les approximations:
- Makov seulement les N derniers mots comptent
- Bigram $P(w_n|w_1^{n-1})\approx P(w_n|w_{n-1})$
- N-Gram $P(w_n|w_1^{n-1})\approx P(w_n|w_{n-N+1}^{n-1})$  
- Phrases en entier modéliser en bigram $P(w_1^{n})\approx \prod_{i=1}^{n}P(w_i|w_{i-1})$ on peut utliser le log pour moins de risquede calcul
- $P(w_1^n)=P(w_1)P(w_2|w_1)....P(w_n|w_{n-1})$


Début et fin de phrase sans approx

Contexte soit la phrase: **the teacher drinks tea**

Si l'on veut calculer la proba on a :
 $$P(\text{the teacher drinks tea})=P(the)P(teacher|the)P(drinks|teacher)P(tea|drinks)$$

Nous n'avons pas le contexte du mot the si l'on souhaite calculer un bigramme par exemple, on ne pourra pas calculer la prbabilité de bigramme P(the) pour faire nos prédictions, donc, ce que nous allons faire est d'ajouter un terme spécial, pour que chaque phrase de notre corpus devienne un bigramme pour lequel nous pouvons calculer les probabilités on ajoutera le caractère: $\text{<s>}$ en début de phrase quand il s'agira de bigramme.

On aura: **$\text{<s> the teacher drinks tea}$**

 $$P(\text{<s> the teacher drinks tea})=P(the|\text{<s>})P(teacher|the)P(drinks|teacher)P(tea|drinks)$$

- Pour la probabiblité d'un trigramme on ajoutera 2 fois $\text{<s>}$  en début de phrase.

- Pour la probabilité d'un Ngrammes on ajoutera N-1 $\text{<s>}$  en début de phrase.

Et pour les fins de phrase

Normalement nous pouvons écrire que
$$P(y|x)=\frac{C(xy)}{\sum_w C(xw)}=\frac{C(xy)}{C(x)}$$

- $\sum_w C(xw)$ le nombre de tous les bigrammes commençant par $x$.
- $C(x)$ le nombre de tous les unigrammes $x$.


Cependant on pourrait avoir un cas ou $\sum_w C(xw)$ et $C(x)$ ne sont pas égaux.

Exemple:

On considère ce corpus de 2 phrases :

- $\text{<s> Lyns drinks chocolate}$
- $\text{<s> John drinks}$

ici $\sum_w C(\text{drinks w})=1$ il s'agit de **drinks chocolate** et $C(x)=2$  car le mot drinks est présent deux fois dans notre corpus...la solution est d'ajouter un tokens de fin de phrase: $\text{<e>}$ à la fin de la seconde phrase.

On considère maintenant ce corpus de 3 phrases de 2 mots:

- $\text{<s> yes no}$
- $\text{<s> yes yes}$
- $\text{<s> no no}$

Regardons les probabilité pour 2 mots avec yes et no: On a $P(\text{<s> yes yes}) = P(yes|s)P(yes|yes)=\frac{C(\text{<s> yes})}{\sum_w C(\text{<s> w})}\frac{C(\text{yes yes})}{\sum_w C(\text{yes w})} = \frac{2}{3}\frac{1}{2}=\frac{1}{3}$ on aura aussi $P(\text{<s> yes no}) =P(\text{<s> no no}) = \frac{1}{3}$ et $ P(\text{<s> no yes}) = 0$ car no yes n'est pas dans notre corpus on remarque aussi en additionant nos 3 premières probabilité que la somme $\sum_{2 mots}P(...)$ vaut 1 on aura aussi $\sum_{3 mots}P(...)=1$, cependant on aimerais bien avoir $\sum_{2 mots}P(...)+\sum_{3 mots}P(...)+......=1$ pour comparer les probabilités de deux phrases de longueurs différentes pour cela on rajoutera $\text{<e>}$ à la fin des phrases pour avoir des proba bien plus petite.

- $\text{<s> Lyns drinks chocolate}$
- $\text{<s> John drinks <e>}$
On a bien  $\sum_w C(\text{drinks w})=2$ et $C(x)=2$

Exemple bigramme

- $\text{<s> Lyns drinks chocolate <e>}$
- $\text{<s> John drinks tea <e>}$
- $\text{<s> Lyns eats chocolate <e>}$

On a
- $P(John|\text{<s>})=\frac{1}{3}$
- $P(chocolate|eats )=\frac{1}{2}$
- $P(\text{<e>}|tea )=\frac{1}{1}$
- $P(Lyns|\text{<s>} )=\frac{2}{3}$

et

$P(\text{<s> Lyns drinks chocolate <e>})=P(Lyns|\text{<s>} )P(drinks|Lyns )P(chocolate|drinks )P(<e>|chocolate )= \frac{2}{3}\frac{1}{2}\frac{1}{2}\frac{2}{2} = \frac{1}{6}$  cette proba est bien inférieur à 1 comme on s'y attendais.

## <a name="mln">Modèle de langage N-gramme</a>

- Supposons que la probabilité du mot suivant ne dépende que du n-gramme précédent.
- Le n-gramme précédent est la série des "n" mots précédents comme vu précédemment.

La probabilité conditionnelle pour le mot à la position "t" dans la phrase, étant donné que les mots qui le précèdent sont $w_{t-1}, w_{t-2} \cdots w_{t-n}$ est :

$$ P(w_t | w_{t-1}\dots w_{t-n}) \tag{1}$$

Vous pouvez estimer cette probabilité en comptant les occurrences de ces séries de mots dans les données de formation (train).
- La probabilité peut être estimée sous la forme d'un rapport, où
- Le numérateur est le nombre de fois que le mot "t" apparaît après les mots t-1 à t-n dans les données de formation.
- Le dénominateur est le nombre de fois que les mots t-1 à t-n apparaissent dans les données de formation.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$

- La fonction $C(\cdots)$ indique le nombre d'occurrences de la séquence donnée.
- La fonction $\hat{P}$ désigne l'estimation de $P$.
- Notez que le dénominateur de l'équation (2) est le nombre d'occurrences des mots $n$ précédents, et le numérateur est la même séquence suivie du mot $w_t$.

Plus tard, vous modifierons l'équation (2) en ajoutant un lissage (smoothing) k, qui évite les erreurs lorsque les comptes de n-grammes sont nuls.

L'équation (2) nous dit que pour estimer les probabilités basées sur les n-grammes, nous avons besoin du nombre de n-grammes (pour le dénominateur) et de (n+1)-grammes (pour le numérateur).

Ensuite nous implémenterons une fonction qui calcule les nombres de n-grammes pour un nombre arbitraire $ n $.

Lors du calcul des nombres de n-grammes, préparons la phrase à l'avance en ajoutant $ n-1 $  marqueurs "<s \>"au début pour indiquer le début de la phrase.
- Par exemple, dans le modèle bi-gramme (N = 2), une séquence d'un tokens de début " <s \>" doit prédire le premier mot d'une phrase.
- Donc, si la phrase est "I like food", modifions-la en "<s \> I like food".
- Préparons également la phrase à compter en ajoutant un token de fin "<e \>" afin que le modèle puisse prédire quand terminer une phrase.

Note technique: Dans cette implémentation, nous stockerons les décomptes sous forme de dictionnaire.
- La clé de chaque paire clé-valeur du dictionnaire est un **tuple** de n mots (et non une liste)
- La valeur de la paire clé-valeur est le nombre d'occurrences.
- La raison de l'utilisation d'un tuple comme clé au lieu d'une liste est qu'une liste en Python est un objet mutable (il peut être modifié après sa création). Un tuple est "immuable", il ne peut donc pas être modifié après sa création. Cela rend un tuple approprié comme type de données pour la clé dans un dictionnaire.

In [ ]:
def count_n_grams(data, n, start_token='<s>', end_token = '<e>') :
    """
    Comptez tous les n-grammes dans les données

    Args :
        data : Liste de listes de mots
        n : nombre de mots dans une séquence

    Retours :
        Un dictionnaire qui fait correspondre un n-uplet de mots à sa fréquence
    """

    # Initialisation du dictionnaire des n-grammes et de leur nombre
    n_grams = {}


    # Passons en revue chaque phrase dans les données
    for sentence in data: # compléter cette ligne

       # ajouter le token de démarrage n-1 fois, et ajouter <e> une fois
        sentence = [start_token] * (n-1) + sentence + [end_token]

        # convertir la liste en tuple
        # Pour que la séquence de mots puisse être utilisée comme
        # une clé dans le dictionnaire
        sentence = tuple(sentence)

        # Utiliser le "i" pour indiquer le début du n-gram
        # de l'index 0
        # au dernier indice où la fin du n-gramme
        # est à l'intérieur de la phrase.
        m = len(sentence) if n==1 else len(sentence)-1
        for i in range(m): # compléter cette ligne

            # Obtenir le n-gram de i à i+n
            n_gram = sentence[i:i+n]

            # vérifier si le n-gram se trouve dans le dictionnaire
            if n_gram in n_grams.keys():

                # Augmenter le nombre de ces n-grammes
                n_grams[n_gram] += 1
            else:
                # Initialiser ce nombre de n-grammes à 1
                n_grams[n_gram] = 1


    return n_grams

In [ ]:
# test
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
print("Uni-gram:")
print(count_n_grams(sentences, 1))
print("Bi-gram:")
print(count_n_grams(sentences, 2))

Uni-gram:
{('i',): 1, ('like',): 2, ('a',): 2, ('cat',): 2, ('<e>',): 2, ('this',): 1, ('dog',): 1, ('is',): 1}
Bi-gram:
{('<s>', 'i'): 1, ('i', 'like'): 1, ('like', 'a'): 2, ('a', 'cat'): 2, ('cat', '<e>'): 2, ('<s>', 'this'): 1, ('this', 'dog'): 1, ('dog', 'is'): 1, ('is', 'like'): 1}


Ensuite, estimons la probabilité d'un mot donné par rapport aux "n" mots précédents en utilisant le nombre de n-grammes.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n)}{C(w_{t-1}\dots w_{t-n})} \tag{2} $$

Cette formule ne fonctionne pas quand le nombre d'un n-gramme est égal à zéro..
- Supposons que nous rencontrions un n-gram qui ne figurait pas dans les données de formation.  
- Alors, l'équation (2) ne peut pas être évaluée (elle devient zéro divisé par zéro).
Exemple bigramme avec ce corpus:

   - $\text{<s> Lyns drinks chocolate <e>}$
   - $\text{<s> John drinks tea <e>}$
   - $\text{<s> Lyns eats chocolate <e>}$

John et eats sont présent dans notre corpus mais le bigramme John eats n'est pas présent.



Une façon de traiter les nombres de zéros est d'ajouter un lissage k.  
- Le **K-smoothing** ajoute une constante positive $k$ à chaque numérateur et $k \times |V|$ au dénominateur, où $|V|$ est le nombre de mots du vocabulaire.

$$ \hat{P}(w_t | w_{t-1}\dots w_{t-n}) = \frac{C(w_{t-1}\dots w_{t-n}, w_n) + k}{C(w_{t-1}\dots w_{t-n}) + k|V|} \tag{3} $$


Pour les n-grammes qui ont un compte de zéro, l'équation (3) devient $\frac{1}{|V|}$.
- Cela signifie que tout n-gramme ayant une valeur nulle a la même probabilité de $\frac{1}{|V|}$.

<a name="back"></a>
Une autre façon de traiter les nombres de zéros est de condidérer n-1 gramme au lieu de n-gramme si le ngramme n'apparait pas il s'agit du **backoff** on peut multiplier par 0.4.  

Dans notre exemple bigramme $P(chocolate|\text{John drinks})=0.4P(chocolate|dinks)$



In [ ]:
# pre-calculated probabilities of all types of n-grams
trigram_probabilities = {('i', 'am', 'happy'): 0}
bigram_probabilities = {( 'am', 'happy'): 0.3}
unigram_probabilities = {'happy': 0.4}

# this is the input trigram we need to estimate
trigram = ('are', 'you', 'happy')

# find the last bigram and unigram of the input
bigram = trigram[1: 3]
unigram = trigram[2]
print(f"besides the trigram {trigram} we also use bigram {bigram} and unigram ({unigram})\n")

# 0.4 is used as an example, experimentally found for web-scale corpuses when using the "stupid" back-off
lambda_factor = 0.4
probability_hat_trigram = 0

# search for first non-zero probability starting with trigram
# to generalize this for any order of n-gram hierarchy,
# you could loop through the probability dictionaries instead of if/else cascade
if trigram not in trigram_probabilities or trigram_probabilities[trigram] == 0:
    print(f"probability for trigram {trigram} not found")

    if bigram not in bigram_probabilities or bigram_probabilities[bigram] == 0:
        print(f"probability for bigram {bigram} not found")

        if unigram in unigram_probabilities:
            print(f"probability for unigram {unigram} found\n")
            probability_hat_trigram = lambda_factor * lambda_factor * unigram_probabilities[unigram]
        else:
            probability_hat_trigram = 0
    else:
        probability_hat_trigram = lambda_factor * bigram_probabilities[bigram]
else:
    probability_hat_trigram = trigram_probabilities[trigram]

print(f"probability for trigram {trigram} estimated as {probability_hat_trigram}")


besides the trigram ('are', 'you', 'happy') we also use bigram ('you', 'happy') and unigram (happy)

probability for trigram ('are', 'you', 'happy') not found
probability for bigram ('you', 'happy') not found
probability for unigram happy found

probability for trigram ('are', 'you', 'happy') estimated as 0.06400000000000002


Une dernière façon est **l'interpolation linéaire** de tous les ordres de n-gramme regardons avec un trigramme un bigramme et un unigramme:

 $$ \hat{P}(chocolate|\text{John drinks}) =0.7P(chocolate|\text{John drinks})+0.2P(chocolate|dinks)+0.1P(chocolate)$$
 $$ \hat{P}(chocolate|\text{John drinks}) =\lambda_1P(chocolate|\text{John drinks})+\lambda_2P(chocolate|dinks)+\lambda_3P(chocolate)$$  $$ \hat{P}(w_n|w_{n-2} w_{n-1}) =\lambda_1P(w_n|w_{n-2} w_{n-1})+\lambda_2P( w_n| w_{n-1})+\lambda_3P(w_n)$$  et $\sum_i\lambda_i=1$ les lambdas sont à régler dans un ensemble de validation en général on les utilise dans de longue séquence .



In [ ]:
# pre-calculated probabilities of all types of n-grams
trigram_probabilities = {('i', 'am', 'happy'): 0.15}
bigram_probabilities = {( 'am', 'happy'): 0.3}
unigram_probabilities = {'happy': 0.4}

# the weights come from optimization on a validation set
lambda_1 = 0.8
lambda_2 = 0.15
lambda_3 = 0.05

# this is the input trigram we need to estimate
trigram = ('i', 'am', 'happy')

# find the last bigram and unigram of the input
bigram = trigram[1: 3]
unigram = trigram[2]
print(f"besides the trigram {trigram} we also use bigram {bigram} and unigram ({unigram})\n")

# in the production code, you would need to check if the probability n-gram dictionary contains the n-gram
probability_hat_trigram = lambda_1 * trigram_probabilities[trigram]
+ lambda_2 * bigram_probabilities[bigram]
+ lambda_3 * unigram_probabilities[unigram]

print(f"estimated probability of the input trigram {trigram} is {probability_hat_trigram}")


besides the trigram ('i', 'am', 'happy') we also use bigram ('am', 'happy') and unigram (happy)

estimated probability of the input trigram ('i', 'am', 'happy') is 0.12


Définissons une fonction qui calcule l'estimation de la probabilité (3) à partir du nombre de n-grammes et d'une constante $k$.

- La fonction prend dans un dictionnaire "n_gram_counts", où la clé est le n-gram et la valeur est le nombre de ce n-gramme.
- La fonction prend également un autre dictionnaire "n_plus1_gram_counts", que vous utiliserez pour trouver le compte du n-gramme précédent plus le mot courant.

In [ ]:
def estimate_probability(word, previous_n_gram,
                         n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0) :
    """
    Estimer les probabilités d'un mot suivant en utilisant le compte n-grammes avec lissage k

    Args :
        word : mot suivant
        previous_n_gram : Une séquence de mots de longueur n
        n_gram_counts : Dictionnaire des comptages de n-grammes
        n_plus1_gram_counts : Dictionnaire des comptages de (n+1)-grammes
        vocabulary_size : nombre de mots dans le vocabulaire
        k : constante positive, paramètre de lissage

    Retours :
        Une probabilité
    """
    # convertir la liste en tuple pour l'utiliser comme clé de dictionnaire
    previous_n_gram = tuple(previous_n_gram)

    ### CODE DE DÉPART ICI (Remplacez les cas de "Aucun" par votre code) ###

    # Fixer le dénominateur
    # Si le n-gram précédent existe dans le dictionnaire des  n-grammes compté,
    # Obtenez son compte.  Sinon, mettez le compte à zéro
    # Utiliser le dictionnaire qui compte les n-grammes
    previous_n_gram_count = n_gram_counts[previous_n_gram] if previous_n_gram in n_gram_counts  else 0

    # Calculer le dénominateur en utilisant le nombre du n gramme précédent
    # et appliquer le k-lissage
    denominator = previous_n_gram_count + k * vocabulary_size

    # Définir n plus 1 gramme comme le n-gramme précédent plus le mot courant comme un tuple
    n_plus1_gram = previous_n_gram + (word,)

    # Réglez le nombre sur le nombre (valeur) du dictionnaire,
    # sinon 0 si pas dans le dictionnaire
    # utiliser le dictionnaire qui compte pour le n-gram plus le mot courant
    n_plus1_gram_count = n_plus1_gram_counts[n_plus1_gram] if n_plus1_gram in n_plus1_gram_counts  else 0

    # Définissez le numérateur en utilisant le nombre de n-grammes + le mot courant,
    # et appliquer un lissage
    numerator = n_plus1_gram_count + k

    # Calculer la probabilité comme le numérateur divisé par le dénominateur
    probability = numerator / denominator


    return probability

In [ ]:
# test
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
tmp_prob = estimate_probability("cat", "a", unigram_counts, bigram_counts, len(unique_words), k=1)

print(f"La probabilité estimée du mot 'cat' étant donné le n-gramme précédent 'a' est: {tmp_prob:.4f}")

La probabilité estimée du mot 'cat' étant donné le n-gramme précédent 'a' est: 0.3333


## <a name="eptm">Estimer les probabilités pour tous les mots</a>

Définissons une fonction qui boucle sur tous les mots du vocabulaire pour calculer les probabilités pour tous les mots possibles.

In [ ]:
def estimate_probabilities(previous_n_gram, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0):
    """
    Estimer les probabilités de mots suivants en utilisant le compte n-grammes avec lissage k

    Args :
        previous_n_gram : Une séquence de mots de longueur n
        n_gram_counts : Dictionnaire du nombre de n-grammes
        n_plus1_gram_counts : Dictionnaire des comptages de (n+1)-grammes
        vocabulaire : Liste de mots
        k : constante positive, paramètre de lissage

    Retours :
        Un dictionnaire qui établit une correspondance entre les mots suivants et la probabilité.
    """

    # convertir la liste en tuple pour l'utiliser comme clé de dictionnaire
    previous_n_gram = tuple(previous_n_gram)

    # ajouter <e> <unk> au vocabulaire
    # <s> n'est pas nécessaire puisqu'il ne doit pas apparaître comme le mot suivant
    vocabulary = vocabulary + ["<e>", "<unk>"]
    vocabulary_size = len(vocabulary)

    probabilities = {}
    for word in vocabulary:
        probability = estimate_probability(word, previous_n_gram,
                                           n_gram_counts, n_plus1_gram_counts,
                                           vocabulary_size, k=k)
        probabilities[word] = probability

    return probabilities

In [ ]:
# test
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
estimate_probabilities("a", unigram_counts, bigram_counts, unique_words, k=1)

{'a': 0.09090909090909091,
 'is': 0.09090909090909091,
 'i': 0.09090909090909091,
 'like': 0.09090909090909091,
 'dog': 0.09090909090909091,
 'this': 0.09090909090909091,
 'cat': 0.2727272727272727,
 '<e>': 0.09090909090909091,
 '<unk>': 0.09090909090909091}

In [ ]:
# utre test
trigram_counts = count_n_grams(sentences, 3)
estimate_probabilities(["<s>", "<s>"], bigram_counts, trigram_counts, unique_words, k=1)

{'a': 0.1111111111111111,
 'is': 0.1111111111111111,
 'i': 0.2222222222222222,
 'like': 0.1111111111111111,
 'dog': 0.1111111111111111,
 'this': 0.2222222222222222,
 'cat': 0.1111111111111111,
 '<e>': 0.1111111111111111,
 '<unk>': 0.1111111111111111}

## <a name="mcp">Matrices de comptage et de probabilité.</a>


Matrice de comptage exemple soit notre phrase:  $\text{ <s> I study I learn <e>}$ on veut représenter le nombre de bigramme par une matrice on obtient la matrice suivante

<table style="width:20%">
                    

<table style="width:40%">

  <tr>
    <td> <b> </b>  </td>
    <td> <b>$\text{<s>}$</b>  </td>
    <td> <b>$\text{<e>}$</b>  </td>
    <td> <b>I</b> </td>
    <td> <b>study</b> </td>
    <td> <b>learn</b> </td>
  </tr>
   <tr>
    <td> <b> $\text{<s>}$</b></td>
     <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>

  </tr>
  <tr>
    <td> <b>$\text{<e>}$</b></td>
   <td> 0</td>
    <td> 0</td>
   <td> 0</td>
    <td> 0</td>
   <td> 0</td>
  </tr>
   
  <tr>
    <td> <b> I </b></td>
    <td> 0</td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 1</td>
  </tr>

  <tr>
    <td> <b> study </b></td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
  </tr>
  
   <tr>
    <td> <b> learn </b></td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
    <td> 0</td>
  </tr>
  

</table>
    
- Les lignes correspondent au premier mot (unique du corpus) du bigram et les colonnes au dernier mot juste après.   
    
- Pour calculer les valeurs de la matrice avec $(N=2)$ pour un mot situé à la position $n$ on calcule : $$C(w_{n-N+1}^{n-1},w_n)$$
    
- à noter que $C(w_{n-N+1}^{n-1},w_n)=C(w_{n-1},w_n)$.
    
**study I est apparu 1 fois dans le corpus, learn I  est apparu 0 fois mais I learn est apparu une fois**
    
- pour la matrice de comptage d'un Ngramme les lignes correspondent au N-1 premier mots (unique du corpus) du Ngramme et les colonnes au dernier mot juste après.
    
    

Matrice de probabilité
    
    
<table style="width:100%">

  <tr>
    <td> <b> </b>  </td>
    <td> <b>$\text{<s>}$</b>  </td>
    <td> <b>$\text{<e>}$</b>  </td>
    <td> <b>I</b> </td>
    <td> <b>study</b> </td>
    <td> <b>learn</b> </td>
    <td> <b>$\sum_wC(w_{n-N+1}^{n-1},w)$</b> </td>
     
  </tr>
   <tr>
    <td> <b> $\text{<s>}$</b></td>
     <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>

  </tr>
  <tr>
    <td> <b>$\text{<e>}$</b></td>
   <td> 0</td>
    <td> 0</td>
   <td> 0</td>
    <td> 0</td>
   <td> 0</td>
     <td> 0</td>
  </tr>
   
  <tr>
    <td> <b> I </b></td>
    <td> 0</td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 1</td>
     <td> 2</td>
  </tr>

  <tr>
    <td> <b> study </b></td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
  </tr>
  
   <tr>
    <td> <b> learn </b></td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
  </tr>
  

</table>

    
    
 <table style="width:40%">

  <tr>
    <td> <b> </b>  </td>
    <td> <b>$\text{<s>}$</b>  </td>
    <td> <b>$\text{<e>}$</b>  </td>
    <td> <b>I</b> </td>
    <td> <b>study</b> </td>
    <td> <b>learn</b> </td>
  </tr>
   <tr>
    <td> <b> $\text{<s>}$</b></td>
     <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>

  </tr>
  <tr>
    <td> <b>$\text{<e>}$</b></td>
   <td> 0</td>
    <td> 0</td>
   <td> 0</td>
    <td> 0</td>
   <td> 0</td>
  </tr>
   
  <tr>
    <td> <b> I </b></td>
    <td> 0</td>
    <td> 0</td>
    <td> 0</td>
    <td> 0.5</td>
    <td> 0.5</td>
  </tr>

  <tr>
    <td> <b> study </b></td>
    <td> 0</td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
  </tr>
  
   <tr>
    <td> <b> learn </b></td>
    <td> 0</td>
    <td> 1</td>
    <td> 0</td>
    <td> 0</td>
    <td> 0</td>
  </tr>
  

</table>
- Les lignes correspondent au premier mot (unique du corpus) du bigram et les colonnes au dernier mot juste après.   
    
- Pour calculer les valeurs de la matrice avec $(N=2)$ pour un mot situé à la position $n$ on calcul la proba suivante : $$P(w_n|w_{n-N+1}^{n-1})=\frac{C(w_{n-N+1}^{n-1},w_n)}{C(w_{n-N+1}^{n-1})}$$
    
- à noter que $C(w_{n-N+1}^{n-1})=\sum_wC(w_{n-N+1}^{n-1},w)$ correspond à la somme de chaque ligne de la matrice de comptage, ici seul le calcul de $C(w_{n-N+1}^{n-1},w_n)=C(w_{n-1},w_n)$ nous intéresse.
    
Exemple: $\text{<s> I learn <e>}$ on a $P(\text{<s> I learn <e>})=P(I|\text{<s>})P(learn|I)P(\text{<e>}|learn)=1x0.5x1=0.5$
    


Comme nous l'avons vu jusqu'à présent, les nombres de n-grammes calculés ci-dessus sont suffisants pour calculer les probabilités du mot suivant.
- Il peut être plus intuitif de les présenter sous forme de matrices de comptage ou de probabilité.
- Les fonctions définies dans les cellules suivantes renvoient des matrices de comptage ou de probabilité.
- Cette fonction est fournie pour vous.

In [ ]:
def make_count_matrix(n_plus1_gram_counts, vocabulary):
    # ajouter <e> <unk> au vocabulaire
    # <s> est omis car il ne doit pas apparaître comme le mot suivant
    vocabulary = vocabulary + ["<e>", "<unk>"]

   # obtenir des n-grammes uniques
    n_grams = []
    for n_plus1_gram in n_plus1_gram_counts.keys():
        n_gram = n_plus1_gram[0:-1]
        n_grams.append(n_gram)
    n_grams = list(set(n_grams))

    # mapping de n-gramme à ligne
    row_index = {n_gram:i for i, n_gram in enumerate(n_grams)}
    # mappage du mot suivant à la colonne
    col_index = {word:j for j, word in enumerate(vocabulary)}

    nrow = len(n_grams)
    ncol = len(vocabulary)
    count_matrix = np.zeros((nrow, ncol))
    for n_plus1_gram, count in n_plus1_gram_counts.items():
        n_gram = n_plus1_gram[0:-1]
        word = n_plus1_gram[-1]
        if word not in vocabulary:
            continue
        i = row_index[n_gram]
        j = col_index[word]
        count_matrix[i, j] = count

    count_matrix = pd.DataFrame(count_matrix, index=n_grams, columns=vocabulary)
    return count_matrix

In [ ]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)

print('nombre de bigramme')
display(make_count_matrix(bigram_counts, unique_words))

nombre de bigramme


,a,is,i,like,dog,this,cat,<e>,<unk>
"(dog,)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(like,)",2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(cat,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(a,)",0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
"(is,)",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
"(this,)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(<s>,)",0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
"(i,)",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# nombre de trigramme
print('\nnombre de trigrammes')
trigram_counts = count_n_grams(sentences, 3)
display(make_count_matrix(trigram_counts, unique_words))


nombre de trigrammes


,a,is,i,like,dog,this,cat,<e>,<unk>
"(dog, is)",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
"(i, like)",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(<s>, i)",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
"(cat,)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(like, a)",0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
"(this, dog)",0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(is, like)",1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"(a, cat)",0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
"(<s>, this)",0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
"(<s>, <s>)",0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


Implémentons une fonction qui calcule les probabilités de chaque mot par rapport au n-gramme précédent, et les stocke sous forme de matrice.

In [ ]:
def make_probability_matrix(n_plus1_gram_counts, vocabulary, k):
    count_matrix = make_count_matrix(n_plus1_gram_counts, unique_words)
    count_matrix += k
    prob_matrix = count_matrix.div(count_matrix.sum(axis=1), axis=0)
    return prob_matrix

In [ ]:
sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))
bigram_counts = count_n_grams(sentences, 2)
print("probabilités de bigramme")
display(make_probability_matrix(bigram_counts, unique_words, k=1))

probabilités de bigramme


,a,is,i,like,dog,this,cat,<e>,<unk>
"(dog,)",0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000
"(like,)",0.272727,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909
"(cat,)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909
"(a,)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909,0.090909
"(is,)",0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000
"(this,)",0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000
"(<s>,)",0.090909,0.090909,0.181818,0.090909,0.090909,0.181818,0.090909,0.090909,0.090909
"(i,)",0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000


In [ ]:
print("probabilités de trigramme")
trigram_counts = count_n_grams(sentences, 3)
display(make_probability_matrix(trigram_counts, unique_words, k=1))

probabilités de trigramme


,a,is,i,like,dog,this,cat,<e>,<unk>
"(dog, is)",0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000
"(i, like)",0.200000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000
"(<s>, i)",0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000
"(cat,)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909
"(like, a)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909,0.090909
"(this, dog)",0.100000,0.200000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000
"(is, like)",0.200000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000,0.100000
"(a, cat)",0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.272727,0.090909
"(<s>, this)",0.100000,0.100000,0.100000,0.100000,0.200000,0.100000,0.100000,0.100000,0.100000
"(<s>, <s>)",0.090909,0.090909,0.181818,0.090909,0.090909,0.181818,0.090909,0.090909,0.090909


Confirm that you obtain the same results as for the `estimate_probabilities` function that you implemented.

## <a name="mlg">Modèle de langage Générative</a>

Soit le Corpus:

- $\text{<s> Lyns drinks chocolate <e>}$   **On choisit $\text{<s> Lyns}$ puis Lyns drinks**
- $\text{<s> John drinks tea <e>}$         **puis  drinks tea  puis $\text{tea <e>}$**
- $\text{<s> Lyns eats chocolate <e>}$


L'algorithme est:
- 1 On choisit une phrase de départ
- 2 On choisit le bigramme suivant commencant par le mot précédent
- 3 On continue jusqu'à ce que $\text{<e>}$ est choisie

Evaluation du modèle de langage :
Dès le début nous avons séparé nos données en Train/Test mais on peut séparer les données textuelles en train test val , val pour le réglage des paramètres et test seulement pour tester le modèle pour un petit corpus on peut prendre respectivement 80%, 10% et 10%, pour un grand corpus 98%,  1% et 1%.
On a 2 méthodes pour divivser le texte:
- choisir des segments de phrase continue longs
- choisir de courte séquence au hasard




## <a name="per">Perplexité</a>


On peut évaluer sur les données test avce la métrique de complexité

Dans cette section, nous allons générer le score de perplexité pour évaluer notre modèle sur le banc d'essai.
- Nous utiliserons également le [**back-off**](#back) lorsque cela sera nécessaire.
- La perplexité est utilisée comme mesure d'évaluation de notre modèle linguistique.
- Pour calculer le score de perplexité de l'ensemble de tests sur un modèle n-gram, nous utilisons la métrique de perplexité :

$$ PP(W) =\sqrt[N]{ \prod_{t=n+1}^N \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } $$

- où $N$ est la taille de la phrase( nombre de mot).
- $n$ est le nombre de mots dans le n-gramme (par exemple 2 pour un bigramme).
- En mathématiques, la numérotation commence à un et non à zéro.

En code python, l'indexation des tableaux commence à zéro, donc le code utilisera des plages pour $t$ selon cette formule :

$$ PP(W) =\sqrt[N]{ \prod_{t=n}^{N-1} \frac{1}{P(w_t | w_{t-n} \cdots w_{t-1})} } $$

Plus les probabilités sont élevées, plus la perplexité sera faible.
- Plus les n-grammes nous renseignent sur la phrase, plus le score de perplexité sera faible et mieux est le modèle, (écrit par un humain).

Calculons le score de perplexité à partir d'une matrice de nombre de N grammes et d'une phrase.

In [ ]:
def calculate_perplexity(sentence, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0):
    """
    Calculer la perplexité pour une liste de phrases

    Args :
        phrase : Liste des chaînes de caractères
        n_gram_counts : Dictionnaire des comptages de (n+1)-grammes
        n_plus1_gram_counts : Dictionnaire des comptages de (n+1)-grammes
        vocabulary_size : nombre de mots uniques dans le vocabulaire
        k : Constante de lissage positive

    Retours :
        Score de perplexité
    """
    # longueur des mots précédents
    n = len(list(n_gram_counts.keys())[0])

    # ajouter <s> et <e>
    sentence = ["<s>"] * n + sentence + ["<e>"]

     # changer la phrase d'une liste à un tuple
    sentence = tuple(sentence)

    # la longueur de la phrase (après avoir ajouté des tokens <s> et <e>)
    N = len(sentence)

    # La variable pi tiendra le produit
    # qui est calculé à l'intérieur de la racine n
    # Mettre à jour le code ci-dessous
    product_pi = 1.0


    # L'indice t varie de n à N - 1, inclus aux deux extrémités
    for t in range(n, N): # complete this line

       # obtenir le n-gram qui précède le mot à la position t
        n_gram = sentence[t-n:t]

        # obtenir le mot à la position t
        word = sentence[t]

        # Estimer la probabilité du mot compte tenu du n-gramme
        # en utilisant les comptes de n-grammes, n-plus1-grammes,
        # taille du vocabulaire, et constante de lissage
        probability = estimate_probability(word,n_gram, n_gram_counts, n_plus1_gram_counts, len(unique_words), k=1)

        # Mettre à jour le produit des probabilités
        # Ce "produit_pi" est un produit cumulatif
        # des facteurs (1/P) qui sont calculés dans la boucle
        product_pi *= 1 / probability

    # Prendre la  racine Nième du produit
    perplexity = product_pi**(1/float(N))


    return perplexity

In [ ]:
# test

sentences = [['i', 'like', 'a', 'cat'],
                 ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)


perplexity_train1 = calculate_perplexity(sentences[0],
                                         unigram_counts, bigram_counts,
                                         len(unique_words), k=1.0)
print(f"Perplexité pour le premier échantillon de train: {perplexity_train1:.4f}")

test_sentence = ['i', 'like', 'a', 'dog']
perplexity_test = calculate_perplexity(test_sentence,
                                       unigram_counts, bigram_counts,
                                       len(unique_words), k=1.0)
print(f"Perplexité pour l'échantillon de test: {perplexity_test:.4f}")

Perplexity for first train sample: 2.6889
Perplexity for test sample: 3.8027


# <a name="4">IV. Construire un système d'Auto-Complétion</a>

Dans cette section, nous combinerons les modèles linguistiques développés jusqu'à présent pour mettre en place un système d'auto-complétion.


Calculons les probabilités pour tous les mots suivants possibles et suggérons le plus probable.
- Cette fonction prend également un argument optionnel `start_with`, qui spécifie les premières lettres des mots suivants.

In [ ]:
def suggest_a_word(previous_tokens, n_gram_counts, n_plus1_gram_counts, vocabulaire, k=1.0, start_with=None) :
    """
    Obtenir une suggestion pour le mot suivant

    Args :
        previous_tokens : La phrase que vous entrez où chaque jeton est un mot. Doit avoir une longueur > n
        n_gram_counts : Dictionnaire des comptages de (n+1)-grammes
        n_plus1_gram_counts : Dictionnaire des comptages de (n+1)-grammes
        vocabulaire : Liste de mots
        k : constante positive, paramètre de lissage
        start_with : Si ce n'est pas le cas, spécifiez les premières lettres du mot suivant

    Retours :
        Un tuple de
          - chaîne du mot suivant le plus probable
          - probabilité correspondante
    """

    # longueur des mots précédents
    n = len(list(n_gram_counts.keys())[0])

    # A partir des mots que l'utilisateur a déjà tapés
    # obtenir les "n" mots les plus récents comme le n-gram précédent
    previous_n_gram = previous_tokens[-n:]

    # Estimer les probabilités que chaque mot du vocabulaire
    # est le mot suivant,
    # étant donné le n-gram précédent, le dictionnaire des n-grammes compte,
    # le dictionnaire de n plus 1 gramme compte, et la constante de lissage
    probabilities = estimate_probabilities(previous_n_gram,
                                           n_gram_counts, n_plus1_gram_counts,
                                           vocabulary, k=k)

    # Initialiser le mot suggéré à Aucun
    # Ce sera le mot avec la plus grande probabilité
    suggestion = None

    # Initialiser la probabilité la plus élevée du mot à 0
    # ce sera fixé à la plus forte probabilité
    # de tous les mots à suggérer
    max_prob = 0

    ### CODE DE DÉPART ICI (Remplacez les cas de "Aucun" par votre code) ###

    # Pour chaque mot et sa probabilité dans le dictionnaire des probabilités :
    for word, prob in probabilities.items():

        # Si la chaîne optionnelle start_with est définie
        if start_with != None:

            # Vérifiez si le début du mot ne correspond pas aux lettres de "start_with".
            if not word.startswith(start_with):

                # si elles ne correspondent pas, sautez ce mot (passez au mot suivant)
                continue

        # Vérifiez si la probabilité de ce mot
        # est supérieure à la probabilité maximale actuelle
        if prob > max_prob:

            # Si oui, gardez ce mot comme meilleure suggestion (jusqu'à présent)
            suggestion = word

            # Sauvegarder la nouvelle probabilité maximale
            max_prob = prob


    return suggestion, max_prob

In [ ]:
# test
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)

previous_tokens = ["i", "like"]
tmp_suggest1 = suggest_a_word(previous_tokens, unigram_counts, bigram_counts, unique_words, k=1.0)
print(f"Les mots précédents sont 'i like',\n\tet le mot suggéré est `{tmp_suggest1[0]}`avec une probabilité de {tmp_suggest1[1]:.4f}")

print()
# testons notre code lors de la configuration de begin_with
tmp_starts_with = 'c'
tmp_suggest2 = suggest_a_word(previous_tokens, unigram_counts, bigram_counts, unique_words, k=1.0, start_with=tmp_starts_with)
print(f"Les mots précédents sont  'i like', la suggestion doit commencer par`{tmp_starts_with}`\n\tet le mot suggéré est `{tmp_suggest2[0]}` avec une probabilité de {tmp_suggest2[1]:.4f}")

Les mots précédents sont 'i like',
	et le mot suggéré est `a`avec une probabilité de 0.0002

Les mots précédents sont  'i like', la suggestion doit commencer par`c`
	et le mot suggéré est `company` avec une probabilité de 0.0001


## <a name="rms">Recevoir de multiples suggestions</a>

On implémente une fonction définie ci-dessous qui permet de boucler sur différents modèles n-grammes pour obtenir de multiples suggestions.

In [ ]:
def get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with=None):
    model_counts = len(n_gram_counts_list)
    suggestions = []
    for i in range(model_counts-1):
        n_gram_counts = n_gram_counts_list[i]
        n_plus1_gram_counts = n_gram_counts_list[i+1]

        suggestion = suggest_a_word(previous_tokens, n_gram_counts,
                                    n_plus1_gram_counts, vocabulary,
                                    k=k, start_with=start_with)
        suggestions.append(suggestion)
    return suggestions

In [ ]:
# test
sentences = [['i', 'like', 'a', 'cat'],
             ['this', 'dog', 'is', 'like', 'a', 'cat']]
unique_words = list(set(sentences[0] + sentences[1]))

unigram_counts = count_n_grams(sentences, 1)
bigram_counts = count_n_grams(sentences, 2)
trigram_counts = count_n_grams(sentences, 3)
quadgram_counts = count_n_grams(sentences, 4)
qintgram_counts = count_n_grams(sentences, 5)

n_gram_counts_list = [unigram_counts, bigram_counts, trigram_counts, quadgram_counts, qintgram_counts]
previous_tokens = ["i", "like"]
tmp_suggest3 = get_suggestions(previous_tokens, n_gram_counts_list, unique_words, k=1.0)

print(f"Les mots précédents sont «i like», les suggestions sont:")
display(tmp_suggest3)

Les mots précédents sont «i like», les suggestions sont:


[('a', 0.00020236087689713323),
 ('a', 0.00013491635186184566),
 ('i', 6.746272684341901e-05),
 ('i', 6.746272684341901e-05)]

## <a name="spmunlv">Suggérer plusieurs mots en utilisant des n-grammes de longueur variable</a>


Nous avons développé tous les éléments de base pour la mise en œuvre de nos propres systèmes de saisie semi-automatique.

Voyons cela avec des n-grammes de longueurs variables (unigrammes, bigrammes, trigrammes, 4-grammes ... 6-grammes).


In [ ]:
n_gram_counts_list = []
for n in range(1, 6):
    print("Calcul des nombres de n-grammes avec n =", n, "...")
    n_model_counts = count_n_grams(train_data_processed, n)
    n_gram_counts_list.append(n_model_counts)

Calcul des nombres de n-grammes avec n = 1 ...
Calcul des nombres de n-grammes avec n = 2 ...
Calcul des nombres de n-grammes avec n = 3 ...
Calcul des nombres de n-grammes avec n = 4 ...
Calcul des nombres de n-grammes avec n = 5 ...


In [ ]:
previous_tokens = ["i", "am", "to"]
tmp_suggest4 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"Les mots précédents sont{previous_tokens}, les suggestions sont:")
display(tmp_suggest4)

Les mots précédents sont['i', 'am', 'to'], les suggestions sont:


[('be', 0.027665685098338604),
 ('have', 0.00013487086115044844),
 ('have', 0.00013490725126475548),
 ('i', 6.746272684341901e-05)]

In [ ]:
previous_tokens = ["i", "want", "to", "go"]
tmp_suggest5 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"Les mots précédents sont{previous_tokens}, les suggestions sont:")
display(tmp_suggest5)

Les mots précédents sont['i', 'want', 'to', 'go'], les suggestions sont:


[('to', 0.014051961029228078),
 ('to', 0.004697942168993581),
 ('to', 0.0009424436216762033),
 ('to', 0.0004044489383215369)]

In [ ]:
previous_tokens = ["hey", "how", "are"]
tmp_suggest6 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"Les mots précédents sont{previous_tokens}, les suggestions sont:")
display(tmp_suggest6)

Les mots précédents sont['hey', 'how', 'are'], les suggestions sont:


[('you', 0.023426812585499317),
 ('you', 0.003559435862995299),
 ('you', 0.00013491635186184566),
 ('i', 6.746272684341901e-05)]

In [ ]:
previous_tokens = ["hey", "how", "are", "you"]
tmp_suggest7 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0)

print(f"Les mots précédents sont {previous_tokens}, les suggestions sont:")
display(tmp_suggest7)

Les mots précédents sont ['hey', 'how', 'are', 'you'], les suggestions sont:


[("'re", 0.023973994311255586),
 ('?', 0.002888465830762161),
 ('?', 0.0016134453781512605),
 ('<e>', 0.00013491635186184566)]

In [ ]:
previous_tokens = ["hey", "how", "are", "you"]
tmp_suggest8 = get_suggestions(previous_tokens, n_gram_counts_list, vocabulary, k=1.0, start_with="d")

print(f"Les mots précédents sont{previous_tokens}, les suggestions sont:")
display(tmp_suggest8)

Les mots précédents sont['hey', 'how', 'are', 'you'], les suggestions sont:


[('do', 0.009020723283218204),
 ('doing', 0.0016411737674785006),
 ('doing', 0.00047058823529411766),
 ('dvd', 6.745817593092283e-05)]


Nous avons terminé ce travail en construisant un modèle d'autocomplétion à l'aide d'un modèle de langage n-gramme !  


# <a name="5">V. Référence </a>

- [Coursera Natural Language Processing with Classification and Vector Spaces](https://www.coursera.org/learn/classification-vector-spaces-in-nlp#syllabus)